# Stage 1: Deterministic Frame Sampling for YOLOv8n Fine-Tuning

**Goal:** 584 training videos → 20 uniformly spaced frames each → 11,680 representative frames.

- Annotations are copied unchanged. Val/test splits are never touched.
- Fully deterministic (no RNG): same input → same output.
- Safety: source dataset is verified unchanged (size + mtime + content hash of labels).
- Output is a self-contained YOLO dataset: `images/train`, `labels/train`, `data.yaml`, `sampling_manifest.csv`.

Run top-to-bottom (Restart & Run All).

## 1. Config

In [1]:
import csv
import filecmp
import hashlib
import re
import shutil
from collections import defaultdict
from pathlib import Path
import statistics as stats


DATASET_ROOT = Path("../Prep-Data")
OUTPUT_ROOT = Path("../Prep-Data-Sampled")
FRAMES_PER_VIDEO = 20
EXPECTED_VIDEOS = 584          # set to None to skip this check
OVERWRITE_OUTPUT = False
IMG_EXTS = {".jpg", ".jpeg", ".png"}

assert FRAMES_PER_VIDEO >= 2, "need >= 2 for endpoint-inclusive spacing"
DATASET_ROOT, OUTPUT_ROOT = DATASET_ROOT.resolve(), OUTPUT_ROOT.resolve()
assert DATASET_ROOT.is_dir(), f"Dataset not found: {DATASET_ROOT}"

# Output must be fully disjoint from the dataset (neither inside nor containing it)
assert not (OUTPUT_ROOT == DATASET_ROOT
            or DATASET_ROOT in OUTPUT_ROOT.parents
            or OUTPUT_ROOT in DATASET_ROOT.parents), "OUTPUT_ROOT overlaps DATASET_ROOT"

print("Dataset:", DATASET_ROOT)
print("Output :", OUTPUT_ROOT)

Dataset: /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Prep-Data
Output : /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Prep-Data-Sampled


## 2. Pre-run snapshot (proves source data is untouched later)

In [2]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def take_snapshot(root: Path) -> dict:
    """rel_path -> (size, mtime_ns, sha256 or None). Hash only labels/yaml (cheap, critical)."""
    snap = {}
    for p in root.rglob("*"):
        if p.is_file():
            rel = p.relative_to(root)
            st = p.stat()
            hashed = p.name == "data.yaml" or "labels" in rel.parts
            snap[rel] = (st.st_size, st.st_mtime_ns, sha256(p) if hashed else None)
    return snap

pre_snapshot = take_snapshot(DATASET_ROOT)

def count_split(split):
    d = DATASET_ROOT / "images" / split
    return sum(1 for f in d.iterdir() if f.is_file()) if d.exists() else 0

pre_counts = {s: count_split(s) for s in ("train", "val", "test")}
print("Pre-run image counts:", pre_counts)

Pre-run image counts: {'train': 206304, 'val': 43350, 'test': 43772}


## 3. Parse filenames → (condition, video_id, frame)
Expected: `<condition>_<video_id>_f<frame>.<ext>`, e.g. `day_1945_f000656.jpg`.

In [3]:
train_img_dir = DATASET_ROOT / "images" / "train"
train_lbl_dir = DATASET_ROOT / "labels" / "train"

train_images = sorted(f for f in train_img_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
assert train_images, f"No training images in {train_img_dir}"

# video_id has no underscores; condition may.
PATTERN = re.compile(r"^(?P<cond>.+)_(?P<vid>[^_]+)_f(?P<frame>\d+)$")

videos = defaultdict(list)          # (cond, vid) -> [(frame, path)]
bad = []
for img in train_images:
    m = PATTERN.match(img.stem)
    if not m:
        bad.append(img.name)
        continue
    videos[(m["cond"], m["vid"])].append((int(m["frame"]), img))

if bad:
    raise ValueError(f"{len(bad)} filenames don't match pattern; first 10: {bad[:10]}")

for k in videos:
    videos[k].sort(key=lambda x: x[0])

print(f"{len(train_images)} train images -> {len(videos)} videos")
if EXPECTED_VIDEOS is not None:
    assert len(videos) == EXPECTED_VIDEOS, f"Expected {EXPECTED_VIDEOS} videos, found {len(videos)}"

206304 train images -> 584 videos


## 4. Inspect video-length distribution (sanity check before sampling)

In [4]:
import statistics as st
lengths = [len(v) for v in videos.values()]
print(f"frames/video  min={min(lengths)}  median={st.median(lengths)}  max={max(lengths)}")

# Uniform-in-index == uniform-in-time only if frame numbers are contiguous
gappy = sum(1 for v in videos.values() if v[-1][0] - v[0][0] + 1 != len(v))
print(f"videos with gaps in frame numbering: {gappy}/{len(videos)}")
if gappy:
    print("WARNING: sampling is uniform over available frames, not the original timeline.")

frames/video  min=135  median=300.0  max=1200
videos with gaps in frame numbering: 0/584


## 5. Deterministic uniform sampling
Index `i*(n-1)//(k-1)` for `i=0..k-1`: includes first and last frame, evenly spaced. Videos with `n < k` keep all frames.

In [5]:
def uniform_indices(n: int, k: int) -> list[int]:
    if n <= k:
        return list(range(n))
    return [(i * (n - 1)) // (k - 1) for i in range(k)]

selected = []      # (cond, vid, frame, path)
short_videos = []
for (cond, vid), frames in videos.items():
    if len(frames) < FRAMES_PER_VIDEO:
        short_videos.append((cond, vid, len(frames)))
    for i in uniform_indices(len(frames), FRAMES_PER_VIDEO):
        f, p = frames[i]
        selected.append((cond, vid, f, p))

assert len({p for *_, p in selected}) == len(selected), "duplicate frames selected"

expected = sum(min(FRAMES_PER_VIDEO, len(v)) for v in videos.values())
assert len(selected) == expected
print(f"Selected {len(selected)} frames from {len(videos)} videos")
print(f"Short videos (< {FRAMES_PER_VIDEO} frames): {len(short_videos)}", short_videos[:5])

Selected 11680 frames from 584 videos
Short videos (< 20 frames): 0 []


## 6. Write the sampled dataset

In [6]:
if OUTPUT_ROOT.exists():
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(f"{OUTPUT_ROOT} exists. Set OVERWRITE_OUTPUT=True or remove it.")
    shutil.rmtree(OUTPUT_ROOT)

out_img_dir = OUTPUT_ROOT / "images" / "train"
out_lbl_dir = OUTPUT_ROOT / "labels" / "train"
out_img_dir.mkdir(parents=True)
out_lbl_dir.mkdir(parents=True)

missing_labels = [p.name for *_, p in selected if not (train_lbl_dir / f"{p.stem}.txt").exists()]
if missing_labels:
    raise FileNotFoundError(f"{len(missing_labels)} selected images lack labels; first 10: {missing_labels[:10]}")

manifest = []
for cond, vid, frame, img in selected:
    lbl = train_lbl_dir / f"{img.stem}.txt"
    shutil.copy2(img, out_img_dir / img.name)
    shutil.copy2(lbl, out_lbl_dir / lbl.name)
    manifest.append([cond, vid, frame, (Path("images/train") / img.name).as_posix()])

with open(OUTPUT_ROOT / "sampling_manifest.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["condition", "video_id", "frame", "image_relpath"])
    w.writerows(manifest)
print("Done copying.")

Done copying.


## 7. `data.yaml` for the next stage
Copies class names from the original `data.yaml` if present. Val/test paths point at the **original untouched** dataset.

In [7]:
import yaml  # pip install pyyaml

src_yaml = DATASET_ROOT / "data.yaml"
names = None
if src_yaml.exists():
    names = yaml.safe_load(src_yaml.read_text()).get("names")
else:
    print("WARNING: no source data.yaml; fill in `names` manually.")

data_cfg = {
    "path": str(OUTPUT_ROOT),
    "train": "images/train",
    "val": str(DATASET_ROOT / "images" / "val"),    # untouched originals (absolute)
    "test": str(DATASET_ROOT / "images" / "test"),
    "names": names,
}
(OUTPUT_ROOT / "data.yaml").write_text(yaml.safe_dump(data_cfg, sort_keys=False))
print((OUTPUT_ROOT / "data.yaml").read_text())

path: /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Prep-Data-Sampled
train: images/train
val: /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Prep-Data/images/val
test: /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Prep-Data/images/test
names:
  0: car
  1: bus
  2: truck
  3: pedestrian
  4: rider
  5: bicycle
  6: traffic light



## 8. Validation

In [8]:
# 1. Per-video counts
per_video = defaultdict(int)
for cond, vid, *_ in selected:
    per_video[(cond, vid)] += 1
for k, frames in videos.items():
    assert per_video[k] == min(FRAMES_PER_VIDEO, len(frames)), f"count mismatch for {k}"

# 2. One-to-one image/label pairing, no stray files
out_imgs = {p.stem: p for p in out_img_dir.iterdir()}
out_lbls = {p.stem: p for p in out_lbl_dir.iterdir()}
assert out_imgs.keys() == out_lbls.keys(), "image/label stems differ"
assert len(out_imgs) == len(selected)

# 3. Byte-identical copies
for stem, img in out_imgs.items():
    assert filecmp.cmp(train_img_dir / img.name, img, shallow=False), f"image differs: {img.name}"
    assert filecmp.cmp(train_lbl_dir / f"{stem}.txt", out_lbls[stem], shallow=False), f"label differs: {stem}"

# 4. Manifest matches disk
with open(OUTPUT_ROOT / "sampling_manifest.csv") as f:
    rows = list(csv.DictReader(f))
assert len(rows) == len(selected)
assert all((OUTPUT_ROOT / r["image_relpath"]).exists() for r in rows)

# 5. Source dataset unchanged (size, mtime, label hashes)
post_snapshot = take_snapshot(DATASET_ROOT)
assert pre_snapshot.keys() == post_snapshot.keys(), "file set changed in source dataset"
changed = [k for k in pre_snapshot if pre_snapshot[k] != post_snapshot[k]]
assert not changed, f"source files modified: {changed[:5]}"

# 6. Splits untouched
post_counts = {s: count_split(s) for s in ("train", "val", "test")}
assert pre_counts == post_counts, (pre_counts, post_counts)

print("ALL CHECKS PASSED")
print(f"Selected {len(selected)} frames from {len(videos)} videos -> {OUTPUT_ROOT}")
print("Split counts unchanged:", post_counts)

ALL CHECKS PASSED
Selected 11680 frames from 584 videos -> /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Prep-Data-Sampled
Split counts unchanged: {'train': 206304, 'val': 43350, 'test': 43772}


## 9. Summary: sampled distribution by condition

In [9]:
by_cond = defaultdict(lambda: [0, 0])   # cond -> [videos, frames]
for (cond, _), frames in videos.items():
    by_cond[cond][0] += 1
    by_cond[cond][1] += min(FRAMES_PER_VIDEO, len(frames))
print(f"{'condition':<20}{'videos':>8}{'frames':>10}")
for c, (v, f) in sorted(by_cond.items()):
    print(f"{c:<20}{v:>8}{f:>10}")

condition             videos    frames
day                      559     11180
evening_low_light          1        20
night                     24       480


In [10]:
# Verify exact image ↔ label pairing in the sampled dataset

for img_path in out_img_dir.iterdir():
    if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    label_path = out_lbl_dir / f"{img_path.stem}.txt"

    assert label_path.exists(), (
        f"Missing label for sampled frame: {img_path.name}"
    )

    assert label_path.stem == img_path.stem, (
        f"Image/label mismatch: {img_path.name} ↔ {label_path.name}"
    )

print("PASS: Every sampled image has the exact corresponding label.")

PASS: Every sampled image has the exact corresponding label.


In [11]:
import csv
import hashlib
import math
import os
import re
import statistics as stats
from collections import Counter, defaultdict
from pathlib import Path

# ---- Stage 1 constants (Phase1-Tune-Improved.ipynb, cell[2] L9-L14) ----------
DATASET_ROOT = Path("../Prep-Data")            # cell[2] L9
OUTPUT_ROOT = Path("../Prep-Data-Sampled")     # cell[2] L10
FRAMES_PER_VIDEO = 20                          # cell[2] L11
IMG_EXTS = {".jpg", ".jpeg", ".png"}           # cell[2] L14 (spec for Stage 2: .jpg/.txt)
# ---- Stage 2 specifics --------------------------------------------------------
NUM_CLASSES = 7   # From the Stage 2 task text (class IDs 0-6). NOT defined in any
                  # Stage 1 file; cross-checked against data.yaml `names` below.
MANIFEST_NAME = "sampling_manifest.csv"        # cell[12] L22
# Stem pattern, Phase1-Tune-Improved.ipynb cell[6] L8 (applied to the stem):
PATTERN = re.compile(r"^(?P<cond>.+)_(?P<vid>[^_]+)_f(?P<frame>\d+)$")

SRC_IMG = DATASET_ROOT / "images" / "train"
SRC_LBL = DATASET_ROOT / "labels" / "train"
OUT_IMG = OUTPUT_ROOT / "images" / "train"
OUT_LBL = OUTPUT_ROOT / "labels" / "train"

EXAMPLES = 10
results = {}    # check name -> "PASS" | "FAIL" | "UNVERIFIABLE" | "ERROR"
details = {}    # check name -> list[str] (offenders / reasons)
warnings_ = []  # free-text warnings


def record(name, status, lines=(), total=None):
    results[name] = status
    lines = list(lines)
    if lines:
        head = f"  [{name}] {status}: {total if total is not None else len(lines)} issue(s)"
        details[name] = [head] + [f"    - {l}" for l in lines[:EXAMPLES]] + (
            [f"    ... +{len(lines) - EXAMPLES} more"] if len(lines) > EXAMPLES else [])


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def scan(dirpath, exts, label):
    """Return ({stem: path}, [stray], [dup_stems]). Never raises on missing/unreadable dirs."""
    files, stray, seen, dups = {}, [], Counter(), []
    if not dirpath.is_dir():
        warnings_.append(f"{label}: directory missing: {dirpath}")
        return files, stray, dups
    try:
        entries = sorted(dirpath.iterdir())
    except OSError as e:
        warnings_.append(f"{label}: unreadable directory {dirpath}: {e}")
        return files, stray, dups
    for p in entries:
        if not p.is_file():
            stray.append(f"{p.name} (not a regular file)")
        elif p.suffix.lower() not in exts:
            stray.append(p.name)
        else:
            seen[p.stem] += 1
            files[p.stem] = p
    dups = [s for s, c in seen.items() if c > 1]
    return files, stray, dups


def parse(stem):
    m = PATTERN.match(stem)
    return (m["cond"], m["vid"], int(m["frame"])) if m else None

### Load: originals and sampled set

In [12]:
src_imgs, src_stray_i, _ = scan(SRC_IMG, IMG_EXTS, "source images")
src_lbls, src_stray_l, _ = scan(SRC_LBL, {".txt"}, "source labels")
out_imgs, out_stray_i, out_dup_i = scan(OUT_IMG, IMG_EXTS, "sampled images")
out_lbls, out_stray_l, out_dup_l = scan(OUT_LBL, {".txt"}, "sampled labels")

for lab, s in (("source images", src_stray_i), ("source labels", src_stray_l),
               ("sampled images", out_stray_i), ("sampled labels", out_stray_l)):
    if s:
        warnings_.append(f"{lab}: {len(s)} stray/non-matching-extension entries, e.g. {s[:3]}")

# Group originals by video (keyed by (cond, vid), same as Stage 1 cell[6])
src_videos, src_unparsed = defaultdict(list), []
for stem, p in src_imgs.items():
    k = parse(stem)
    (src_videos[k[:2]].append((k[2], p)) if k else src_unparsed.append(stem))
for v in src_videos.values():
    v.sort(key=lambda x: x[0])

out_videos, out_unparsed = defaultdict(list), []
for stem, p in out_imgs.items():
    k = parse(stem)
    (out_videos[k[:2]].append((k[2], p)) if k else out_unparsed.append(stem))
for v in out_videos.values():
    v.sort(key=lambda x: x[0])

for lab, u in (("source", src_unparsed), ("sampled", out_unparsed)):
    if u:
        warnings_.append(f"{lab}: {len(u)} image filenames don't match PATTERN, e.g. {u[:3]}")

usable = bool(src_imgs) and bool(out_imgs)
if not usable:
    warnings_.append("Source or sampled image set is empty/missing; most checks cannot run.")

### 1. Source videos (computed, not hardcoded)


In [13]:
n_src_videos, n_out_videos = len(src_videos), len(out_videos)
missing_v = sorted(set(src_videos) - set(out_videos))
extra_v = sorted(set(out_videos) - set(src_videos))
c1 = [f"video in source but absent from sample: {v}" for v in missing_v] + \
     [f"video in sample but absent from source: {v}" for v in extra_v]
record("Source videos", "PASS" if usable and not c1 else "FAIL", c1, len(c1))

### 2. Per-video sample count == min(20, frames available)


In [14]:
c2, per_video_counts = [], []
for k, frames in src_videos.items():
    exp, act = min(FRAMES_PER_VIDEO, len(frames)), len(out_videos.get(k, []))
    per_video_counts.append(act)
    if exp != act:
        c2.append(f"{k}: expected {exp}, actual {act}")
expected_images = sum(min(FRAMES_PER_VIDEO, len(f)) for f in src_videos.values())
record("Per-video count", "PASS" if usable and not c2 else "FAIL", c2, len(c2))

### 3-4. Pairing (strict by stem) and totals


In [15]:
orph_img = sorted(set(out_imgs) - set(out_lbls))
orph_lbl = sorted(set(out_lbls) - set(out_imgs))
c3 = [f"image without label: {s}" for s in orph_img] + \
     [f"label without image: {s}" for s in orph_lbl]
if len(out_imgs) != len(out_lbls):
    c3.append(f"totals differ: {len(out_imgs)} images vs {len(out_lbls)} labels")
record("Image-label pairing", "PASS" if usable and not c3 else "FAIL", c3, len(c3))

### 5. Annotation validity (5 fields, class 0..NUM_CLASSES-1, ranges, finite)


In [16]:
def check_label_file(path):
    """Return (errors, is_empty). Unreadable -> error, never raises."""
    try:
        text = path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError) as e:
        return [f"{path.name}: unreadable ({e})"], False
    lines = [(i, l) for i, l in enumerate(text.splitlines(), 1) if l.strip()]
    if not lines:
        return [], True
    errs = []
    for i, line in lines:
        f = line.split()
        if len(f) != 5:
            errs.append(f"{path.name}:{i}: {len(f)} fields (need 5)")
            continue
        try:
            cid_f = float(f[0])
            vals = [float(x) for x in f[1:]]
        except ValueError:
            errs.append(f"{path.name}:{i}: non-numeric field")
            continue
        if not all(math.isfinite(x) for x in [cid_f] + vals):
            errs.append(f"{path.name}:{i}: NaN/inf")
            continue
        if not (cid_f.is_integer() and re.fullmatch(r"[+-]?\d+", f[0])):
            errs.append(f"{path.name}:{i}: class id {f[0]!r} not an integer")
            continue
        if not 0 <= int(f[0]) < NUM_CLASSES:
            errs.append(f"{path.name}:{i}: class id {f[0]} outside 0-{NUM_CLASSES - 1}")
        xc, yc, w, h = vals
        if not (0 <= xc <= 1 and 0 <= yc <= 1):
            errs.append(f"{path.name}:{i}: x/y center out of [0,1]: {xc},{yc}")
        if not (0 < w <= 1 and 0 < h <= 1):
            errs.append(f"{path.name}:{i}: width/height out of (0,1]: {w},{h}")
    return errs, False


c5, empty_labels, rows_total = [], [], 0
for stem, p in sorted(out_lbls.items()):
    errs, empty = check_label_file(p)
    c5 += errs
    if empty:
        empty_labels.append(stem)
    elif not errs:
        rows_total += 1
# Empty files are reported separately; valid YOLO "no objects". Not counted as FAIL.
record("Annotation validity", "PASS" if out_lbls and not c5 else "FAIL", c5, len(c5))
if empty_labels:
    warnings_.append(f"{len(empty_labels)} empty label files (valid YOLO 'no objects'; "
                     f"{100 * len(empty_labels) / max(len(out_lbls), 1):.2f}% of sampled), "
                     f"e.g. {empty_labels[:3]}. Review if this share looks high.")

# Cross-check class count against data.yaml if present and PyYAML importable (read-only).
try:
    import yaml  # noqa: WPS433 - only if already installed in the repo env
    for yp in (OUTPUT_ROOT / "data.yaml", DATASET_ROOT / "data.yaml"):
        if yp.is_file():
            names = yaml.safe_load(yp.read_text()).get("names")
            n = len(names) if names is not None else None
            if n is not None and n != NUM_CLASSES:
                warnings_.append(f"{yp}: `names` has {n} classes but NUM_CLASSES={NUM_CLASSES}")
except ImportError:
    warnings_.append("PyYAML not importable: could not cross-check NUM_CLASSES vs data.yaml.")
except Exception as e:  # noqa: BLE001
    warnings_.append(f"data.yaml check skipped: {e}")

### 6. Duplicate stems in the sampled set


In [17]:
c6 = [f"duplicate image stem: {s}" for s in out_dup_i] + \
     [f"duplicate label stem: {s}" for s in out_dup_l]
record("Duplicate frames", "PASS" if out_imgs and not c6 else "FAIL", c6, len(c6))

### 7. Deterministic: re-run Stage 1 selection in memory (no writes)


In [18]:
def uniform_indices(n, k):           # Phase1-Tune-Improved.ipynb cell[10] L1-L4
    if n <= k:
        return list(range(n))
    return [(i * (n - 1)) // (k - 1) for i in range(k)]


expected_stems = set()
for k, frames in src_videos.items():
    for i in uniform_indices(len(frames), FRAMES_PER_VIDEO):
        expected_stems.add(frames[i][1].stem)
actual_stems = set(out_imgs)
c7 = [f"expected but missing from sample: {s}" for s in sorted(expected_stems - actual_stems)] + \
     [f"in sample but not expected: {s}" for s in sorted(actual_stems - expected_stems)]
record("Deterministic sampling", "PASS" if usable and not c7 else "FAIL", c7, len(c7))

### 8. Uniform sampling
#### **Definition (from Stage 1 algorithm):** for a video with n >= k original frames,
#### the sampled frames must sit at original-list indices i*(n-1)//(k-1), i=0..k-1.
#### Consequences verified directly: (a) first and last original frame are included;
#### (b) consecutive *index* gaps differ by at most 1 (floor spacing => gaps are
#### floor or ceil of (n-1)/(k-1)); tolerance = 0 for (a), <=1 for (b).
#### For n < k all frames must be kept. Uniformity is over *available* frames (index
#### space), not real time; frame-number gaps are reported as a warning, not a failure.

In [19]:
c8, gappy_videos = [], 0
for k, frames in src_videos.items():
    n = len(frames)
    stem_to_idx = {p.stem: i for i, (_, p) in enumerate(frames)}
    got = sorted(stem_to_idx[p.stem] for _, p in out_videos.get(k, []) if p.stem in stem_to_idx)
    if frames[-1][0] - frames[0][0] + 1 != n:
        gappy_videos += 1
    if n < FRAMES_PER_VIDEO:
        if got != list(range(n)):
            c8.append(f"{k}: n={n}<k, expected all frames kept")
        continue
    if got != uniform_indices(n, FRAMES_PER_VIDEO):
        c8.append(f"{k}: sampled indices deviate from evenly-spaced definition")
        continue
    gaps = [b - a for a, b in zip(got, got[1:])]
    lo, hi = (n - 1) // (FRAMES_PER_VIDEO - 1), -(-(n - 1) // (FRAMES_PER_VIDEO - 1))
    if got[0] != 0 or got[-1] != n - 1 or any(g < lo or g > hi for g in gaps):
        c8.append(f"{k}: endpoints/gaps outside [{lo},{hi}] (gaps {min(gaps)}..{max(gaps)})")
record("Uniform sampling", "PASS" if usable and not c8 else "FAIL", c8, len(c8))
if gappy_videos:
    warnings_.append(f"{gappy_videos}/{n_src_videos} videos have gaps in original frame numbering: "
                     "uniformity holds over available frames, not original timeline.")

### 9. Label preservation (SHA-256 vs original train label)


In [20]:
c9 = []
for stem, p in sorted(out_lbls.items()):
    sp = src_lbls.get(stem)
    if sp is None:
        c9.append(f"{stem}: no original train label")
        continue
    try:
        if sha256(p) != sha256(sp):
            c9.append(f"{stem}: hash differs from original")
    except OSError as e:
        c9.append(f"{stem}: unreadable ({e})")
record("Label preservation", "PASS" if out_lbls and not c9 else "FAIL", c9, len(c9))


### 10. Val/test unchanged


In [21]:
def split_count(split):
    d = DATASET_ROOT / "images" / split
    if not d.is_dir():
        return None
    try:
        return sum(1 for f in d.iterdir() if f.is_file())
    except OSError:
        return None


BASELINE_CANDIDATES = [OUTPUT_ROOT / "val_test_baseline.json", DATASET_ROOT / "val_test_baseline.json",
                       OUTPUT_ROOT / "baseline.json"]
baseline = next((p for p in BASELINE_CANDIDATES if p.is_file()), None)
cur = {s: split_count(s) for s in ("val", "test")}
print("Current split counts (informational, not a check):", cur)
for s in ("val", "test"):
    if baseline is None:
        results[f"{s} data unchanged"] = "UNVERIFIABLE"
        details[f"{s} data unchanged"] = [
            f"  [{s}] UNVERIFIABLE: no persisted baseline found (looked for "
            f"{[str(p) for p in BASELINE_CANDIDATES]}); current count={cur[s]}"]
    else:
        # A baseline schema was never defined by Stage 1. Refuse to guess its format.
        results[f"{s} data unchanged"] = "UNVERIFIABLE"
        details[f"{s} data unchanged"] = [
            f"  [{s}] UNVERIFIABLE: found {baseline} but its schema is undefined by Stage 1; "
            "not guessing a format"]

Current split counts (informational, not a check): {'val': 43350, 'test': 43772}


### Report


In [22]:
def line(label, status):
    print(f"{label + ':':<26}{status}")


print("\nStage 2 Validation")
print("------------------")
print(f"Source videos: {n_src_videos}  (sampled videos: {n_out_videos})")
print(f"Sampled images: {len(out_imgs)}")
print(f"Sampled labels: {len(out_lbls)}")
print(f"Expected images: {expected_images}        (computed: sum of min({FRAMES_PER_VIDEO}, frames) per video)")
if per_video_counts:
    print(f"Images per video: min {min(per_video_counts)} / max {max(per_video_counts)} / "
          f"mean {stats.mean(per_video_counts):.2f}")
else:
    print("Images per video: n/a")
print(f"Empty label files: {len(empty_labels)} (reported, not a failure)\n")

order = [("Source videos", "Source videos"), ("Per-video count", "Per-video count"),
         ("Image-label pairing", "Image-label pairing"), ("Annotation validity", "Annotation validity"),
         ("Duplicate frames", "Duplicate frames"), ("Deterministic sampling", "Deterministic sampling"),
         ("Uniform sampling", "Uniform sampling"), ("Label preservation", "Label preservation"),
         ("Validation data unchanged", "val data unchanged"), ("Test data unchanged", "test data unchanged")]
for label, key in order:
    line(label, results.get(key, "ERROR"))

statuses = [results.get(k, "ERROR") for _, k in order]
overall = "PASS" if all(s == "PASS" for s in statuses) else "FAIL"
ready = "YES" if overall == "PASS" else "NO"
print(f"\nOverall: {overall}")
print(f"Ready for Stage 3: {ready}")
if "UNVERIFIABLE" in statuses:
    print("  (UNVERIFIABLE item(s) present: Ready=NO until you explicitly accept the gap.)")

failed = [k for k, s in results.items() if s in ("FAIL", "ERROR")]
if failed or any(v for v in details.values()):
    print("\nOffending examples (max %d each)" % EXAMPLES)
    for k in list(results):
        if k in details:
            print("\n".join(details[k]))

print("\nExceptions / warnings")
print("---------------------")
ASSUMPTIONS = [
    "NUM_CLASSES=7 comes from the Stage 2 task text, not from any Stage 1 file "
    "(Phase1-Tune-Improved.ipynb never sets a class count; it only copies `names` at cell[14] L6).",
    "Paths are relative (Phase1-Tune-Improved.ipynb cell[2] L9-L10); run from the same working dir as Stage 1 (Model/).",
    "Check 1: 'compare with sampled set' = same video-key sets; the 584 figure is only in Stage 1 "
    "(cell[2] L12 / original cell[18] L2) and is intentionally not asserted here.",
    "Check 8: uniformity is defined over sorted original index space per Stage 1 cell[10] L1-L4, tolerance 0 on "
    "endpoints/indices, gaps within floor/ceil((n-1)/(k-1)).",
    "Stage 1 uses no RNG/seed (cell[10]); 'deterministic' = pure function of sorted frame lists.",
    "Stage 1 wrote no val/test baseline (only in-memory snapshot, cell[4]); checks 10a/10b are UNVERIFIABLE.",
]
for w in warnings_ + ASSUMPTIONS:
    print(" -", w)



Stage 2 Validation
------------------
Source videos: 584  (sampled videos: 584)
Sampled images: 11680
Sampled labels: 11680
Expected images: 11680        (computed: sum of min(20, frames) per video)
Images per video: min 20 / max 20 / mean 20.00
Empty label files: 152 (reported, not a failure)

Source videos:            PASS
Per-video count:          PASS
Image-label pairing:      PASS
Annotation validity:      PASS
Duplicate frames:         PASS
Deterministic sampling:   PASS
Uniform sampling:         PASS
Label preservation:       PASS
Validation data unchanged:UNVERIFIABLE
Test data unchanged:      UNVERIFIABLE

Overall: FAIL
Ready for Stage 3: NO
  (UNVERIFIABLE item(s) present: Ready=NO until you explicitly accept the gap.)

Offending examples (max 10 each)
  [val] UNVERIFIABLE: no persisted baseline found (looked for ['../Prep-Data-Sampled/val_test_baseline.json', '../Prep-Data/val_test_baseline.json', '../Prep-Data-Sampled/baseline.json']); current count=43350
  [test] UNVERIFI

In [23]:
import csv
import importlib.util
import math
import os
import platform
import re
import shutil
import time
import traceback
from collections import Counter
from pathlib import Path

MPS_FALLBACK = True   # PYTORCH_ENABLE_MPS_FALLBACK: unsupported MPS ops run on CPU instead of crashing. Must be set before torch import.
if MPS_FALLBACK:
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

In [24]:
# ---- Stage 3 notebook configuration ----
# Resolve the project root from the notebook's CURRENT WORKING DIRECTORY.
# This avoids ../ paths breaking when Jupyter is launched from a different folder.

import sys
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "Prep-Data").is_dir() and (candidate / "Prep-Data-Sampled").is_dir() and (candidate / "yolov8n.pt").is_file():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate MINOR-PROJECT root. Expected Prep-Data/, "
        "Prep-Data-Sampled/, and yolov8n.pt in the same project root."
    )

DATASET_ROOT = PROJECT_ROOT / "Prep-Data"
SAMPLED_ROOT = PROJECT_ROOT / "Prep-Data-Sampled"
MANIFEST_NAME = "sampling_manifest.csv"
IMG_EXTS = {".jpg", ".jpeg", ".png"}
PATTERN = re.compile(r"^(?P<cond>.+)_(?P<vid>[^_]+)_f(?P<frame>\d+)$")

EXPECTED_TRAIN_VIDEOS = 584
EXPECTED_TRAIN_IMAGES = 11_680
EXPECTED_VAL_IMAGES = 43_350
EXPECTED_TEST_IMAGES = 43_772
EXPECTED_CLASS_NAMES = ["car", "bus", "truck", "pedestrian", "rider", "bicycle", "traffic light"]

SMOKE_EPOCHS = 2
DEVICE = "mps"

# Exact values used by the stock YOLOv8n baseline.
STOCK_WEIGHTS = PROJECT_ROOT / "yolov8n.pt"
IMGSZ = 640

SMOKE_ROOT = PROJECT_ROOT / "Stage3-Smoke"
VAL_SUBSET_SIZE = 2_000
RUN_NAME = "smoke"

# Keep this False for a clean smoke-test configuration. If Albumentations is
# installed in the notebook kernel, preflight will verify it can be disabled.
ALLOW_ALBUMENTATIONS = False

NUM_CLASSES = len(EXPECTED_CLASS_NAMES)
RUNS_DIR = SMOKE_ROOT / "runs"
SMOKE_YAML = SMOKE_ROOT / "data_smoke.yaml"
NEEDS_REVIEW = 10

print("Project root :", PROJECT_ROOT)
print("Python       :", sys.executable)
print("Weights      :", STOCK_WEIGHTS)
print("Image size   :", IMGSZ)


Project root : /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT
Python       : /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/.venv/bin/python
Weights      : /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/yolov8n.pt
Image size   : 640


In [25]:
CHECKS = []   # (name, status, detail); status in PASS / FAIL / WARN
S = {}        # shared state between cells


def record(name, status, detail=""):
    CHECKS.append((name, status, detail))
    print(f"[{status:<4}] {name}" + (f"  -  {detail}" if detail else ""))


def head(items, n=NEEDS_REVIEW):
    items = list(items)
    return "; ".join(map(str, items[:n])) + (f"; ... +{len(items) - n} more" if len(items) > n else "")


def scan_names(d, exts):
    """File names in d with an allowed suffix (names only, nothing opened). None if d is missing."""
    if not d.is_dir():
        return None
    with os.scandir(d) as it:
        return sorted(e.name for e in it if e.is_file() and Path(e.name).suffix.lower() in exts)


def video_key(stem):
    m = PATTERN.match(stem)
    return (m["cond"], m["vid"]) if m else None


def load_names(yaml_path):
    import yaml   # PyYAML is an Ultralytics dependency
    if not yaml_path.is_file():
        return None
    n = yaml.safe_load(yaml_path.read_text()).get("names")
    if isinstance(n, list):
        return {i: str(x) for i, x in enumerate(n)}
    if isinstance(n, dict):
        return {int(k): str(v) for k, v in n.items()}
    return None


def check_label_file(path):
    """Return (errors, is_empty, Counter of class ids from fully valid rows). Never raises."""
    try:
        lines = [(i, l) for i, l in enumerate(path.read_text(encoding="utf-8").splitlines(), 1) if l.strip()]
    except (OSError, UnicodeDecodeError) as e:
        return [f"{path.name}: unreadable ({e})"], False, Counter()
    if not lines:
        return [], True, Counter()
    errs, cls = [], Counter()
    for i, line in lines:
        f = line.split()
        if len(f) != 5:
            errs.append(f"{path.name}:{i}: {len(f)} fields")
            continue
        try:
            vals = [float(x) for x in f[1:]]
            cid = float(f[0])
        except ValueError:
            errs.append(f"{path.name}:{i}: non-numeric")
            continue
        if not all(math.isfinite(x) for x in vals + [cid]):
            errs.append(f"{path.name}:{i}: NaN/inf")
            continue
        if not re.fullmatch(r"\d+", f[0]) or int(f[0]) >= NUM_CLASSES:
            errs.append(f"{path.name}:{i}: bad class id {f[0]!r} (need int 0-{NUM_CLASSES - 1})")
            continue
        xc, yc, w, h = vals
        if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
            errs.append(f"{path.name}:{i}: box out of range {vals}")
            continue
        cls[int(f[0])] += 1
    return errs, False, cls


def scan_labels(label_dir, stems):
    errs, empties, counts = [], [], Counter()
    for s in stems:
        e, empty, c = check_label_file(label_dir / f"{s}.txt")
        errs += e
        counts += c
        if empty:
            empties.append(s)
    return errs, empties, counts


def source_snapshot():
    """Read-only fingerprint of the ORIGINAL dataset: split file counts + existing *.cache files (Ultralytics writes those)."""
    snap = {}
    for kind, exts in (("images", IMG_EXTS), ("labels", {".txt"})):
        for split in ("train", "val", "test"):
            names = scan_names(DATASET_ROOT / kind / split, exts)
            snap[f"{kind}/{split}"] = None if names is None else len(names)
    lbl = DATASET_ROOT / "labels"
    snap["caches"] = sorted((p.name, p.stat().st_mtime_ns) for p in lbl.glob("*.cache")) if lbl.is_dir() else []
    return snap


In [26]:
def preflight_env():
    record("python", "PASS", f"{platform.python_version()} | {sys.executable}")

    # The notebook must actually be using the environment where Ultralytics is installed.
    venv_expected = PROJECT_ROOT / ".venv" / "bin" / "python"
    if venv_expected.is_file():
        same_python = Path(sys.executable).resolve() == venv_expected.resolve()
        record("Jupyter kernel", "PASS" if same_python else "WARN",
               f"kernel={Path(sys.executable).resolve()} | project venv={venv_expected.resolve()}")
        if not same_python:
            print("  IMPORTANT: Ultralytics may be installed in .venv but this notebook kernel is different.")
            print("  Select the project's .venv Python/Jupyter kernel before running training.")
    else:
        record("project .venv", "WARN", f"not found at {venv_expected}")

    S["torch_ok"] = False
    try:
        import torch
        S["torch_ok"] = True
        record("torch", "PASS", torch.__version__)
        if DEVICE == "mps":
            if not (torch.backends.mps.is_built() and torch.backends.mps.is_available()):
                record("MPS available", "FAIL", "torch.backends.mps not built/available")
            else:
                x = torch.randn(64, 64, device="mps")
                finite = math.isfinite(float((x @ x).sum()))
                record("MPS available", "PASS" if finite else "FAIL", "matmul on mps ran")
    except ImportError:
        record("torch", "FAIL", "not installed")
    except Exception as e:
        record("MPS available", "FAIL", f"{type(e).__name__}: {e}")

    if importlib.util.find_spec("ultralytics") is None:
        record("ultralytics", "FAIL", f"not installed in notebook kernel: {sys.executable}")
    else:
        import ultralytics
        record("ultralytics", "PASS", ultralytics.__version__)
        S["ultralytics_version"] = ultralytics.__version__

    if importlib.util.find_spec("albumentations") is not None:
        import inspect
        from ultralytics.data.augment import Albumentations
        sig = inspect.signature(Albumentations.__init__)
        can_disable = "transforms" in sig.parameters
        if can_disable and not ALLOW_ALBUMENTATIONS:
            record("Albumentations control", "PASS", "API allows transforms override (will pass augmentations=[])")
        elif ALLOW_ALBUMENTATIONS:
            record("Albumentations control", "WARN", "ALLOW_ALBUMENTATIONS is True, augmentations will be applied")
        else:
            record("Albumentations control", "FAIL", "Cannot verify API to disable Albumentations")
    else:
        record("Albumentations control", "PASS", "not installed")

    ok = STOCK_WEIGHTS.is_file() and IMGSZ == 640

    record(
        "baseline config",
        "PASS" if ok else "FAIL",
        f"weights={STOCK_WEIGHTS} | imgsz={IMGSZ}"
        + ("" if ok else " | expected stock weights file + imgsz=640")
    )
        


preflight_env()


[PASS] python  -  3.14.6 | /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/.venv/bin/python
[PASS] Jupyter kernel  -  kernel=/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 | project venv=/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14
[PASS] torch  -  2.14.0
[PASS] MPS available  -  matmul on mps ran
[PASS] ultralytics  -  8.4.149
[PASS] Albumentations control  -  not installed
[PASS] baseline config  -  weights=/Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/yolov8n.pt | imgsz=640


In [27]:
def preflight_data():
    tr_i = scan_names(SAMPLED_ROOT / "images" / "train", IMG_EXTS)
    tr_l = scan_names(SAMPLED_ROOT / "labels" / "train", {".txt"})
    if not tr_i or tr_l is None:
        record("sampled train present", "FAIL", f"missing/empty under {SAMPLED_ROOT}")
        return
    tr_stems_i, tr_stems_l = {Path(n).stem for n in tr_i}, {Path(n).stem for n in tr_l}
    S["train_stems"] = sorted(tr_stems_i)
    ok = len(tr_i) == len(tr_l) == EXPECTED_TRAIN_IMAGES and tr_stems_i == tr_stems_l
    record("train images/labels", "PASS" if ok else "FAIL",
           f"{len(tr_i)} images, {len(tr_l)} labels, expected {EXPECTED_TRAIN_IMAGES}; "
           f"orphans: {head(sorted(tr_stems_i ^ tr_stems_l))}" if not ok else f"{len(tr_i)} pairs")

    mf = SAMPLED_ROOT / MANIFEST_NAME
    if mf.is_file():
        with open(mf, newline="") as f:
            man = {Path(r["image_relpath"]).stem for r in csv.DictReader(f)}
        record("train == Stage 1 manifest", "PASS" if man == tr_stems_i else "FAIL",
               f"{len(man)} manifest rows vs {len(tr_stems_i)} images")
    else:
        record("train == Stage 1 manifest", "WARN", f"{mf} not found")

    keys = {"train": {video_key(s) for s in tr_stems_i}}
    unparsed = [s for s in tr_stems_i if video_key(s) is None]
    record("train filenames parse", "PASS" if not unparsed else "FAIL", head(unparsed))
    record("train video count", "PASS" if len(keys["train"] - {None}) == EXPECTED_TRAIN_VIDEOS else "FAIL",
           f"{len(keys['train'] - {None})} videos, expected {EXPECTED_TRAIN_VIDEOS}")

    for split, exp in (("val", EXPECTED_VAL_IMAGES), ("test", EXPECTED_TEST_IMAGES)):
        names = scan_names(DATASET_ROOT / "images" / split, IMG_EXTS)
        if names is None:
            record(f"{split} images present", "FAIL", "directory missing")
            continue
        record(f"{split} image count", "PASS" if len(names) == exp else "FAIL", f"{len(names)}, expected {exp}")
        keys[split] = {video_key(Path(n).stem) for n in names}
        if split == "val":
            S["val_files"] = {Path(n).stem: n for n in names}

    for split in ("val", "test"):
        if split in keys:
            overlap = sorted(k for k in keys["train"] & keys[split] if k)
            record(f"train/{split} share no video", "PASS" if not overlap else "FAIL",
                   f"{len(overlap)} shared (cond, video_id) keys: {head(overlap)}. Leakage, or per-split ID reuse - investigate."
                   if overlap else "disjoint")
    if "val" in keys and "test" in keys:
        overlap = sorted(k for k in keys["val"] & keys["test"] if k)
        record("val/test share no video", "PASS" if not overlap else "FAIL", head(overlap) if overlap else "disjoint")


if __name__ == "__main__":
    preflight_data()

[PASS] train images/labels  -  11680 pairs
[PASS] train == Stage 1 manifest  -  11680 manifest rows vs 11680 images
[PASS] train filenames parse
[PASS] train video count  -  584 videos, expected 584
[PASS] val image count  -  43350, expected 43350
[PASS] test image count  -  43772, expected 43772
[PASS] train/val share no video  -  disjoint
[PASS] train/test share no video  -  disjoint
[PASS] val/test share no video  -  disjoint


In [28]:
def preflight_classes_and_labels():
    src, smp = load_names(DATASET_ROOT / "data.yaml"), load_names(SAMPLED_ROOT / "data.yaml")
    names = src or smp
    if names is None:
        record("class names", "FAIL", f"no `names` in {DATASET_ROOT / 'data.yaml'} or {SAMPLED_ROOT / 'data.yaml'}")
        return

    S["names"] = names
    if src and smp and src != smp:
        record("source vs sampled yaml names", "FAIL", f"{src} != {smp}")

    ok = sorted(names) == list(range(NUM_CLASSES)) and set(names.values()) == set(EXPECTED_CLASS_NAMES)
    order_note = ("order matches expected list" if [names.get(i) for i in range(NUM_CLASSES)] == EXPECTED_CLASS_NAMES
                  else "order differs from expected list")
    record("class names == expected 7", "PASS" if ok else "FAIL", f"{names} | {order_note}")

    if "train_stems" in S:
        errs, empties, cnt = scan_labels(SAMPLED_ROOT / "labels" / "train", S["train_stems"])
        record("train label validity", "PASS" if not errs else "FAIL",
               f"{len(errs)} bad rows: {head(errs)}" if errs else "all rows valid")
        S["train_class_counts"] = cnt

        missing = [names.get(i, i) for i in range(NUM_CLASSES) if cnt[i] == 0]
        # A class with zero annotations is a dataset/evaluation warning, not a reason
        # to declare the entire Stage 3 smoke-test plumbing broken.
        record("every class in train", "PASS" if not missing else "WARN",
               f"zero instances: {missing}; dataset currently contains no annotations for these classes"
               if missing else "all declared classes have training instances")

        print("  train instances per class:",
              {f"{i}:{names.get(i)}": cnt[i] for i in range(NUM_CLASSES)},
              f"| empty label files: {len(empties)}/{len(S['train_stems'])}")

    vf = S.get("val_files")
    if vf:
        lbl = scan_names(DATASET_ROOT / "labels" / "val", {".txt"}) or []
        lstems = {Path(n).stem for n in lbl}
        orphans = sorted(set(vf) ^ lstems)
        record("val image/label pairing", "PASS" if not orphans else "FAIL",
               f"{len(orphans)} orphans: {head(orphans)}" if orphans else f"{len(vf)} pairs")

        errs, empties, cnt = scan_labels(DATASET_ROOT / "labels" / "val", sorted(lstems & set(vf)))
        record("val label validity", "PASS" if not errs else "FAIL",
               f"{len(errs)} bad rows: {head(errs)}" if errs else "all rows valid")

        print("  val instances per class:",
              {f"{i}:{names.get(i)}": cnt[i] for i in range(NUM_CLASSES)},
              f"| empty label files: {len(empties)}")
        absent_val = [names.get(i, i) for i in range(NUM_CLASSES) if cnt[i] == 0]
        if absent_val:
            record("classes represented in full val", "WARN",
                   f"no validation instances for {absent_val}; AP for them is undefined")


preflight_classes_and_labels()


[PASS] class names == expected 7  -  {0: 'car', 1: 'bus', 2: 'truck', 3: 'pedestrian', 4: 'rider', 5: 'bicycle', 6: 'traffic light'} | order matches expected list
[PASS] train label validity  -  all rows valid
[WARN] every class in train  -  zero instances: ['rider']; dataset currently contains no annotations for these classes
  train instances per class: {'0:car': 80660, '1:bus': 1483, '2:truck': 4672, '3:pedestrian': 8528, '4:rider': 0, '5:bicycle': 199, '6:traffic light': 7263} | empty label files: 152/11680
[PASS] val image/label pairing  -  43350 pairs
[PASS] val label validity  -  all rows valid
  val instances per class: {'0:car': 289283, '1:bus': 7021, '2:truck': 18312, '3:pedestrian': 41519, '4:rider': 0, '5:bicycle': 764, '6:traffic light': 25118} | empty label files: 968
[WARN] classes represented in full val  -  no validation instances for ['rider']; AP for them is undefined


In [29]:
def build_smoke_val():
    import yaml
    roots = [SMOKE_ROOT.resolve(), DATASET_ROOT.resolve(), SAMPLED_ROOT.resolve()]
    s, d, p = roots
    if s == d or s == p or d in s.parents or p in s.parents or s in d.parents or s in p.parents:
        record("SMOKE_ROOT disjoint", "FAIL", "SMOKE_ROOT overlaps a dataset dir")
        return
    vf, names = S.get("val_files"), S.get("names")
    if not vf or not names:
        record("smoke val build", "FAIL", "val files or class names unavailable (preflight failed)")
        return
    stems = sorted(set(vf) & {Path(n).stem for n in (scan_names(DATASET_ROOT / "labels" / "val", {".txt"}) or [])})
    n, k = len(stems), min(VAL_SUBSET_SIZE, len(stems))
    idx = [(i * (n - 1)) // (k - 1) for i in range(k)] if k > 1 else [0]
    chosen = [stems[i] for i in idx]

    for sub in ("images", "labels"):
        shutil.rmtree(SMOKE_ROOT / sub / "val", ignore_errors=True)   # only the subset dirs, never SMOKE_ROOT itself
        (SMOKE_ROOT / sub / "val").mkdir(parents=True)
    for stem in chosen:
        shutil.copy2(DATASET_ROOT / "images" / "val" / vf[stem], SMOKE_ROOT / "images" / "val" / vf[stem])
        shutil.copy2(DATASET_ROOT / "labels" / "val" / f"{stem}.txt", SMOKE_ROOT / "labels" / "val" / f"{stem}.txt")

    errs, empties, cnt = scan_labels(SMOKE_ROOT / "labels" / "val", chosen)
    S["val_subset_counts"], S["val_subset_stems"] = cnt, chosen
    S["val_subset_first_img"] = SMOKE_ROOT / "images" / "val" / vf[chosen[0]]
    record("smoke val subset", "PASS" if not errs and len(chosen) == k else "FAIL", f"{len(chosen)} of {n} val pairs copied")
    absent = [names.get(i, i) for i in range(NUM_CLASSES) if cnt[i] == 0]
    record("every class in smoke val", "PASS" if not absent else "WARN",
           f"no instances of {absent} - per-class val AP for them will be undefined" if absent else str(dict(sorted(cnt.items()))))

    data_cfg = {"path": str(SAMPLED_ROOT.resolve()), "train": "images/train",
                "val": str((SMOKE_ROOT / "images" / "val").resolve()), "names": {int(i): v for i, v in sorted(names.items())}}
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    SMOKE_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
    record("smoke yaml has no test key", "PASS" if "test" not in data_cfg else "FAIL", str(SMOKE_YAML))
    S["src_snapshot"] = source_snapshot()   # fingerprint of the original dataset BEFORE training


if __name__ == "__main__":
    build_smoke_val()

[PASS] smoke val subset  -  2000 of 43350 val pairs copied
[WARN] every class in smoke val  -  no instances of ['rider'] - per-class val AP for them will be undefined
[PASS] smoke yaml has no test key  -  /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Stage3-Smoke/data_smoke.yaml


In [30]:
def run_training():

    from ultralytics import YOLO

    t0 = time.perf_counter()

    if not STOCK_WEIGHTS.is_file():
        raise FileNotFoundError(
            f"Stock weights not found: {STOCK_WEIGHTS}"
        )

    model = YOLO(str(STOCK_WEIGHTS))

    train_kwargs = {
        "data": str(SMOKE_YAML.resolve()),
        "epochs": SMOKE_EPOCHS,
        "imgsz": IMGSZ,
        "device": DEVICE,
        "project": str(RUNS_DIR.resolve()),
        "name": RUN_NAME,
    }

    # Explicitly disable Ultralytics Albumentations.
    # This prevents automatic CLAHE/Blur/MedianBlur/ToGray
    # from being introduced during the detector-only smoke test.
    if not ALLOW_ALBUMENTATIONS:
        train_kwargs["augmentations"] = []

    print("\n--- Training Configuration (Verified Control) ---")

    for k, v in train_kwargs.items():
        print(f"  {k}: {v}")

    print("-------------------------------------------------\n")

    model.train(**train_kwargs)

    S["train_seconds"] = time.perf_counter() - t0
    S["save_dir"] = Path(model.trainer.save_dir)


# Notebook cell: run directly
# No __main__ guard required.

if any(status == "FAIL" for _, status, _ in CHECKS):

    print("Preflight has FAILs -> training skipped.")

else:

    try:

        run_training()

        record(
            "training completed",
            "PASS",
            f"{S['train_seconds']:.0f}s total "
            f"(incl. per-epoch val + final val), "
            f"run dir {S['save_dir']}"
        )

    except Exception as e:

        record(
            "training completed",
            "FAIL",
            f"{type(e).__name__}: {e}"
        )

        traceback.print_exc()


--- Training Configuration (Verified Control) ---
  data: /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Stage3-Smoke/data_smoke.yaml
  epochs: 2
  imgsz: 640
  device: mps
  project: /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Stage3-Smoke/runs
  name: smoke
  augmentations: []
-------------------------------------------------

New https://pypi.org/project/ultralytics/8.4.166 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.149 🚀 Python-3.14.6 torch-2.14.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[], auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Stage3-Smoke/data_smoke.yaml, degrees=0.0, deterministic=True, device=mps, dfl

/Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:1072: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic suppo

        1/2      5.27G     0.9447      1.176     0.9516        274        640: 100% ━━━━━━━━━━━━ 730/730 1.3it/s 9:310.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 1.8it/s 34.3s0.5ss
                   all       2000      17626      0.592      0.588       0.62      0.452

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:1072: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic suppo

        2/2      5.27G     0.8665     0.7795     0.9272        272        640: 100% ━━━━━━━━━━━━ 730/730 2.6s/it 31:500.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 1.9it/s 33.2s0.5ss
                   all       2000      17626      0.669      0.653      0.714      0.542

2 epochs completed in 0.710 hours.
Optimizer stripped from /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Stage3-Smoke/runs/smoke/weights/last.pt, 6.2MB
Optimizer stripped from /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Stage3-Smoke/runs/smoke/weights/best.pt, 6.2MB

Validating /Users/rhythemsabharwalgmail.com/Desktop/MINOR-PROJECT/Stage3-Smoke/runs/smoke/weights/best.pt...
Ultralytics 8.4.149 🚀 Python-3.14.6 torch-2.14.0 MPS (Apple M4)
Model summary (fused): 72 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/6